# 🎮 Free Minecraft Server on Kaggle

Automatically set up a Minecraft (Java Edition) server on Kaggle's free compute.

**Run the cells in order.** The two that matter most for a live server:

* **Cell 9** starts Paper through the control agent and waits for Paper's own
  readiness line. It does not report success just because a Java process exists.
* **Cell 15** is the runtime supervisor. It blocks on purpose so this kernel stays
  alive while the server should run, prints real health information, and returns
  only when you press **Stop** in the dashboard.

**How stopping works:** press Stop in the dashboard. The agent records
`desired_state = STOPPED` first, sends Minecraft's own `stop` so the world saves,
waits for the JVM to exit (SIGTERM, then SIGKILL only if it must), and stops the
Playit tunnel. Nothing restarts the server after an intentional Stop.

**If Minecraft crashes on its own,** the supervisor detects the exit, records the
exit code and a log tail, and restarts it with exponential backoff up to a bounded
number of attempts. After that the server is reported as **CRASHED** in the
dashboard with the reason — it is never restarted in a loop.

**Important limitations:**
- Kaggle notebooks run for max ~9 hours per session
- NO permanent storage in `/kaggle/working` — back the world up to Google Drive
- Internet access may be restricted depending on your Kaggle account settings
- No inbound connections — the Playit tunnel provides the public address
- RAM/CPU limited by your Kaggle tier (typically 2 cores, 13-16 GB RAM free tier)
- If Kaggle ends the session, the server stops with it. No component here can
  keep it alive across a terminated runtime, and none claims to.


In [ ]:
# Cell 1: Environment Diagnostics
import platform
import subprocess
import shutil
import os
import sys

print("=" * 60)
print("MINECRAFT SERVER ENVIRONMENT DIAGNOSTICS")
print("=" * 60)

# OS Info
print(f"\n🖥️  Operating System: {platform.system()} {platform.release()}")
print(f"📱 Architecture: {platform.machine()}")
print(f"🐍 Python Version: {platform.python_version()}")

# CPU Cores
try:
    cpu_count = os.cpu_count()
    print(f"⚡ CPU Cores: {cpu_count or 'Unknown'}")
except:
    print("⚠️  Could not detect CPU count")

# RAM (in GB)
try:
    with open('/proc/meminfo', 'r') as f:
        mem_info = f.read()
    total_kb = int([line for line in mem_info.split('\n') if line.startswith('MemTotal:')][0].split()[1])
    total_gb = total_kb / (1024**2)
    print(f"💾 Total RAM: {total_gb:.1f} GB")
except Exception as e:
    print(f"⚠️  RAM detection failed: {e}")

# Disk Space
try:
    total, used, free = shutil.disk_usage('/')
    print(f"💽 Disk - Total: {total/(1024**3):.1f}GB, Free: {free/(1024**3):.1f}GB")
except Exception as e:
    print(f"⚠️  Disk detection failed: {e}")

# Check Java — Minecraft 1.21.x requires Java 21+. A live run on Kaggle shows
# the image already ships OpenJDK 21, so no install is normally needed; if the
# version is lower, say so with the exact remedy instead of failing later.
print("\n🔍 Checking Java...")
import re as _re
import shutil as _shutil

_java_exe = _shutil.which("java")
_java_line, _java_major = None, None
if _java_exe:
    _res = subprocess.run([_java_exe, "-version"], capture_output=True, text=True)
    _out = (_res.stderr or "") + (_res.stdout or "")
    _java_line = _out.strip().splitlines()[0] if _out.strip() else None
if _java_line:
    _m = _re.search(r'"(?:1\.)?(\d+)', _java_line)
    _java_major = int(_m.group(1)) if _m else None

if _java_major is None:
    print("❌ Java NOT found — it is required to run the Minecraft server.")
    print("   Install Java 21, then re-run this cell:")
    print("   !apt-get update && apt-get install -y openjdk-21-jdk-headless")
elif _java_major < 21:
    print(f"⚠️  Java {_java_major} is too old for Minecraft 1.21.x (need 21+): {_java_line}")
    print("   Install Java 21, then re-run this cell:")
    print("   !apt-get update && apt-get install -y openjdk-21-jdk-headless")
    print("   (do not use `default-jdk` — it can install an older JVM)")
else:
    print(f"✅ Java {_java_major} found (Minecraft 1.21.x needs 21+): {_java_line}")

# Internet Test
print("\n🌐 Testing Internet Connectivity...")
try:
    import urllib.request
    urllib.request.urlopen('https://www.minecraft.net', timeout=10)
    print("✅ Internet access OK")
except Exception as e:
    print(f"❌ Internet access issue: {e}")

print("\n" + "=" * 60)
print("NOTE: Kaggle sessions are TEMPORARY. World data is lost when")
print("the notebook stops. Use Google Drive backup to persist your world!")
print("=" * 60)

In [ ]:
# Cell 2: Configuration and Dependency Installation
import os
import subprocess

print("=" * 60)
print("CONFIGURATION")
print("=" * 60)

# ===== EDIT THESE SETTINGS =====
SERVER_EDITION = "java"  # "bedrock" or "java"
SERVER_VERSION = "1.21.4"  # Minecraft version
EULA_ACCEPTED = True  # Set to True after reading https://minecraft.net/eula
VIEW_DISTANCE = 6  # Lower = less lag, higher = see farther
SIMULATION_DISTANCE = 4  # Mob tick distance
MAX_PLAYERS = 5
DIFFICULTY = "normal"  # peaceful, easy, normal, hard
SEED = ""  # Leave empty for random, or set a specific seed
MOTD = "Kaggle Minecraft Server"
ONLINE_MODE = True  # Set False for offline/cracked clients

# Memory allocation (in MB) - be conservative!
# For Kaggle free tier, use 1024-2048MB max
JAVA_XMS = 1024  # Initial heap
JAVA_XMX = 2048  # Max heap (DO NOT exceed available RAM)

# Google Drive Backup Settings
ENABLE_DRIVE_BACKUP = True  # Set to True to enable Google Drive persistence
BACKUP_INTERVAL_MINUTES = 30  # How often to backup during server runtime
MAX_BACKUPS_TO_KEEP = 5  # Number of recent backups to retain
DRY_RUN = False  # Set True to test without actual Drive operations

print(f"📋 Server Edition: {SERVER_EDITION}")
print(f"📋 View Distance: {VIEW_DISTANCE}")
print(f"📋 Max Players: {MAX_PLAYERS}")
print(f"📋 Difficulty: {DIFFICULTY}")
print(f"📋 Seed: {SEED or '(random)'}")
print(f"📋 EULA: {'Accepted' if EULA_ACCEPTED else 'NOT Accepted - server will not start'}")
print(f"📋 Google Drive Backup: {'Enabled' if ENABLE_DRIVE_BACKUP else 'Disabled'}")

# Create server directory
SERVER_DIR = "/kaggle/working/minecraft_server"
os.makedirs(SERVER_DIR, exist_ok=True)
print(f"\n📁 Server directory: {SERVER_DIR}")

# Install required packages for Drive backup
if ENABLE_DRIVE_BACKUP:
    print("\n📦 Installing Google Drive backup dependencies...")
    !pip install google-api-python-client google-auth google-auth-oauthlib google-auth-httplib2 -q
    print("✅ Dependencies installed")

In [ ]:
# Cell 3: Google Drive Authentication
import os
import json

print("=" * 60)
print("GOOGLE DRIVE AUTHENTICATION")
print("=" * 60)

# Import the backup module
import sys
sys.path.insert(0, '/kaggle/working')

# Check for credentials (supports both naming conventions)
# Uses same credential names as your n8n setup for consistency
client_id = os.environ.get('GDRIVE_CLIENT_ID') or os.environ.get('GOOGLE_CLIENT_ID') or os.environ.get('GOOGLE_OAUTH_CLIENT_ID', '')
client_secret = os.environ.get('GDRIVE_CLIENT_SECRET') or os.environ.get('GOOGLE_CLIENT_SECRET') or os.environ.get('GOOGLE_OAUTH_CLIENT_SECRET', '')

if ENABLE_DRIVE_BACKUP:
    print("\n📋 Google Drive Setup Requirements:")
    print("   Same credentials as your n8n setup (if you have them):")
    print("   - GDRIVE_CLIENT_ID / GDRIVE_CLIENT_SECRET")
    print("   - Or: GOOGLE_CLIENT_ID / GOOGLE_CLIENT_SECRET")
    print("   - Or: GOOGLE_OAUTH_CLIENT_ID / GOOGLE_OAUTH_CLIENT_SECRET")
    print("\n   If starting fresh:")
    print("   1. Create Google Cloud project at https://console.cloud.google.com")
    print("   2. Enable Google Drive API")
    print("   3. Create OAuth 2.0 credentials (Desktop app type)")
    print("   4. Add credentials to Kaggle Secrets:")
    print("      - GDRIVE_CLIENT_ID (or GOOGLE_CLIENT_ID)")
    print("      - GDRIVE_CLIENT_SECRET (or GOOGLE_CLIENT_SECRET)")
    print("   5. Run this cell to authenticate")
    
    if client_id and client_secret:
        print("\n✅ Credentials found in Kaggle Secrets")
        print(f"   Using: {os.environ.get('GDRIVE_CLIENT_ID') and 'GDRIVE_*' or os.environ.get('GOOGLE_OAUTH_CLIENT_ID') and 'GOOGLE_OAUTH_*' or 'GOOGLE_CLIENT_*'}")
        print("   Attempting authentication...")
        
        # Import and run authentication
        try:
            from drive_backup import GoogleDriveBackup
            drive = GoogleDriveBackup(dry_run=DRY_RUN)
            auth_success = drive.authenticate()
            
            if auth_success:
                print("\n✅ Google Drive authenticated successfully!")
                print("   You can now use backup/restore functions.")
            else:
                print("\n⚠️  Authentication failed or not completed")
                print("   See KAGGLE_MINECRAFT_GUIDE.md for setup help")
        except ImportError:
            print("⚠️  drive_backup.py not found")
            print("   Make sure the file is in the same directory as this notebook")
        except Exception as e:
            print(f"❌ Authentication error: {e}")
    else:
        print("\n⚠️  No Google credentials found in Kaggle Secrets")
        print("   Add credentials (see above for naming options)")
        print("   See guide for setup instructions")
else:
    print("\n⏭️  Google Drive backup disabled")
    print("   Set ENABLE_DRIVE_BACKUP=True to enable")

In [ ]:
# Cell 4: Drive Connection Test & Backup Folder Setup
if ENABLE_DRIVE_BACKUP:
    print("=" * 60)
    print("TESTING GOOGLE DRIVE CONNECTION")
    print("=" * 60)
    
    try:
        from drive_backup import GoogleDriveBackup
        
        drive = GoogleDriveBackup(dry_run=DRY_RUN)
        
        # Test authentication if not already done
        if not drive.authenticated:
            drive.authenticate()
        
        # Test connection
        if drive.test_connection():
            print("\n✅ Google Drive connection working!")
            
            # List existing backups
            backups = drive.list_backups()
            if backups:
                print(f"\n📋 Found {len(backups)} existing backup(s)")
            else:
                print("\n📁 No existing backups found")
                print("   First backup will be created after server starts")
        else:
            print("\n⚠️  Drive connection test failed")
            print("   Backups will be saved locally only")
            
    except Exception as e:
        print(f"❌ Error: {e}")
        print("   Continuing without Drive backup")
else:
    print("=" * 60)
    print("GOOGLE DRIVE CONNECTION SKIPPED")
    print("=" * 60)
    print("\n   Enable ENABLE_DRIVE_BACKUP to use Drive persistence")

In [ ]:
# Cell 5: World restoration from Google Drive (run before starting the server)
#
# Safety rules, in order of priority:
#   * an existing world is NEVER overwritten unless you explicitly ask
#   * a missing world is only restored when Drive backup is enabled AND a backup
#     exists; the restore path itself takes a safety copy first
#   * a failure here leaves whatever world is on disk untouched
import os

print("=" * 60)
print("WORLD RESTORATION FROM GOOGLE DRIVE")
print("=" * 60)

SERVER_DIR = os.environ.get("SERVER_DIR", "/kaggle/working/minecraft_server")

if not globals().get("ENABLE_DRIVE_BACKUP"):
    print("\nGoogle Drive restoration skipped (ENABLE_DRIVE_BACKUP is off).")
else:
    try:
        from drive_backup import GoogleDriveBackup

        drive = GoogleDriveBackup(dry_run=globals().get("DRY_RUN", False))

        if not drive.authenticated:
            print("\nNot authenticated with Google Drive.")
            print("   Run Cell 3 (Authentication) first.")
        else:
            existing_world = os.path.join(SERVER_DIR, "world")
            if os.path.isdir(existing_world):
                print("\nExisting world detected - it will be PRESERVED.")
                print("   To replace it from a backup instead:")
                print("     1. Set RESTORE_FROM_DRIVE = True below")
                print("     2. Set FORCE_REPLACE = True to overwrite without prompting")
                print("     3. Run this cell again")

                RESTORE_FROM_DRIVE = False   # set True to restore on purpose
                FORCE_REPLACE = False        # set True to skip the confirmation

                if RESTORE_FROM_DRIVE:
                    if drive.restore_backup(force_replace=FORCE_REPLACE):
                        print("\nWorld restored from Google Drive.")
                    else:
                        print("\nRestore did not complete; the existing world is unchanged.")
                else:
                    print("\nKeeping the existing world.")
            else:
                print("\nNo existing world found. Attempting to restore from Drive...")
                if drive.restore_backup():
                    print("\nWorld restored from Google Drive.")
                else:
                    print("\nNo backup to restore, or the restore failed.")
                    print("   A brand-new world will be generated when the server starts.")
                    print("   If that is not what you want, stop here and check Drive first.")
    except Exception as e:
        print(f"\nWorld restoration could not run: {e}")
        print("   Nothing on disk was modified; continuing with what is already there.")

In [ ]:
# Cell 6: Paper JAR download and verification
#
# Downloads the official Paper build for the configured Minecraft version and
# installs it as paper.jar. All of the work happens in backend/paper_download.py
# so the notebook, the agent, and the tests share ONE verification path:
#   * official PaperMC API only (no mirrors)
#   * build number, size, and published SHA-256 read from the same response
#   * temp file -> HTTP status, announced size, byte limit, ZIP integrity,
#     SHA-256 -> atomic os.replace into place
#   * an existing paper.jar is NEVER replaced unless PAPER_FORCE_DOWNLOAD=1
import os
import sys

SERVER_DIR = os.environ.get("SERVER_DIR", "/kaggle/working/minecraft_server")
os.environ["SERVER_DIR"] = SERVER_DIR
os.makedirs(SERVER_DIR, exist_ok=True)

# The repo may sit beside the notebook or be cloned into /kaggle/working.
for _candidate in (os.environ.get("PANEL_DIR"), "/kaggle/working/rahee-world", "/kaggle/working"):
    if _candidate and os.path.isdir(os.path.join(_candidate, "backend")):
        if _candidate not in sys.path:
            sys.path.insert(0, _candidate)
        break

from backend.paper_download import download_paper_jar, minecraft_version

print("=" * 60)
print("DOWNLOADING PAPER SERVER JAR")
print("=" * 60)
print(f"Minecraft version : {minecraft_version()}")
print(f"Server directory  : {SERVER_DIR}")

_force = os.environ.get("PAPER_FORCE_DOWNLOAD", "0").strip().lower() in ("1", "true", "yes")
_result = download_paper_jar(force=_force)
_state = _result.get("state")

if _state == "skipped":
    print(f"Keeping the existing paper.jar: {_result['result'].get('message')}")
elif _state == "success":
    _r = _result["result"]
    print("paper.jar downloaded and verified")
    print(f"  build    : {_r.get('build')}")
    print(f"  size     : {_r.get('size_bytes')} bytes")
    print(f"  sha256   : {_r.get('sha256')}")
    _verified = ("verified against the PaperMC API" if _r.get("checksum_verified")
                 else "the API published no checksum for this build; computed locally")
    print(f"  checksum : {_verified}")
else:
    raise RuntimeError(f"Paper download failed: {_result.get('error')}")

print("=" * 60)

In [ ]:
# Cell 7: Server configuration (eula.txt + server.properties)
#
# Writes the two files Paper needs before it can boot. The values match the
# Stage 1-3 panel and the Stage 4 world/version managers, which read `level-name`
# and `server-port` back out of server.properties.
import os

SERVER_DIR = os.environ.get("SERVER_DIR", "/kaggle/working/minecraft_server")
EULA_ACCEPTED = True   # set after reading https://minecraft.net/eula
SERVER_PORT = int(os.environ.get("SERVER_PORT", "25565"))
LEVEL_NAME = os.environ.get("LEVEL_NAME", "world")

print("=" * 60)
print("GENERATING SERVER CONFIGURATION")
print("=" * 60)

eula_path = os.path.join(SERVER_DIR, "eula.txt")
with open(eula_path, "w") as f:
    f.write("#By changing the setting below to TRUE you are indicating your agreement to our EULA.\n")
    f.write("#You must reboot the server after changing this option.\n")
    f.write("eula=%s\n" % ("true" if EULA_ACCEPTED else "false"))

if EULA_ACCEPTED:
    print("EULA accepted (eula=true)")
    print("   Remember: use the dashboard 'Stop server' button so the world saves")
else:
    print("EULA NOT accepted - the server will refuse to start.")
    print("   Read https://minecraft.net/eula, then set EULA_ACCEPTED = True")

properties = [
    "#Minecraft server properties",
    "server-name=Kaggle Minecraft Server",
    "motd=Kaggle Minecraft Server",
    "gamemode=survival",
    "difficulty=normal",
    "hardcore=false",
    "level-name=%s" % LEVEL_NAME,
    "level-type=minecraft\\:normal",
    "generate-structures=true",
    "allow-nether=true",
    "spawn-npcs=true",
    "spawn-animals=true",
    "spawn-monsters=true",
    "spawn-protection=16",
    "online-mode=true",
    "pvp=true",
    "white-list=false",
    "enforce-whitelist=false",
    "max-players=5",
    "view-distance=6",
    "simulation-distance=4",
    "network-compression-threshold=256",
    "player-idle-timeout=0",
    "max-world-size=29999984",
    "enable-command-block=false",
    "enable-query=false",
    "enable-rcon=false",
    "op-permission-level=4",
    "server-ip=",
    "server-port=%d" % SERVER_PORT,
    "use-native-transport=true",
    "max-tick-time=60000",
    "snooper-enabled=false",
]
props_path = os.path.join(SERVER_DIR, "server.properties")
with open(props_path, "w") as f:
    f.write("\n".join(properties) + "\n")
print("server.properties created (%d settings)" % len(properties))

print("\nConfiguration summary:")
print("   - main world : %s" % LEVEL_NAME)
print("   - port       : %d" % SERVER_PORT)
print("   - players    : 5")
print("   - view dist  : 6")
print("   - online mode: true")
print("\n" + "=" * 60)

In [ ]:
# Cell 8: Start the RaheeCraft control agent
#
# Runs the Kaggle-side agent in a background thread inside this kernel. The
# agent is the SINGLE owner of the server lifecycle: the dashboard's
# start/stop/restart/console buttons are executed here through
# agent.actions.ActionExecutor, which is what makes the console channel and the
# dashboard controls actually work.
#
# Required Kaggle Secrets:
#   BACKEND_URL    - your panel URL, e.g. https://panel.example.com
#   AGENT_ID       - value registered by the panel (or generated once and saved)
#   AGENT_SECRET   - matching secret for that agent id
#
# Do not run a second server starter alongside this cell: two Paper processes
# must never open the same world directory.
import os
import sys
import threading
import time

SERVER_DIR = os.environ.get("SERVER_DIR", "/kaggle/working/minecraft_server")
os.environ["SERVER_DIR"] = SERVER_DIR

# Give Paper a realistic readiness window on the first run, when it is
# generating a world from scratch.
os.environ.setdefault("START_TIMEOUT_SECONDS", "600")

for _candidate in (os.environ.get("PANEL_DIR"), "/kaggle/working/rahee-world", "/kaggle/working"):
    if _candidate and os.path.isdir(os.path.join(_candidate, "agent")):
        if _candidate not in sys.path:
            sys.path.insert(0, _candidate)
        break

for _key in ("BACKEND_URL", "AGENT_ID", "AGENT_SECRET", "CORS_ORIGIN"):
    if not os.environ.get(_key):
        try:
            from kaggle_secrets import UserSecretsClient
            _value = UserSecretsClient().get_secret(_key)
            if _value:
                os.environ[_key] = _value
        except Exception:
            pass

from agent.config import config as agent_config

_missing = [k for k in ("AGENT_ID", "AGENT_SECRET", "BACKEND_URL") if not os.environ.get(k)]
if _missing:
    raise RuntimeError(
        "Missing Kaggle Secrets: %s. Add them in the notebook Secrets panel "
        "(and make sure the notebook is allowed to access them)." % ", ".join(_missing)
    )

print("=" * 60)
print("STARTING RAHEECRAFT CONTROL AGENT")
print("=" * 60)
print(f"Agent ID    : {os.environ['AGENT_ID'][:8]}...")
print(f"Backend     : {os.environ['BACKEND_URL']}")
print(f"Server dir  : {SERVER_DIR}")

from agent.main import ControlAgent

_agent = ControlAgent()
_agent_thread = threading.Thread(target=_agent.run, name="raheecraft-agent", daemon=True)
_agent_thread.start()
time.sleep(5)

if _agent_thread.is_alive():
    print("Agent thread running. The dashboard can now control the server.")
else:
    raise RuntimeError("Agent thread exited immediately - check the log output above.")
print("=" * 60)

In [ ]:
# Cell 9: Start the Paper server (through the agent, so the console works)
#
# Uses the same ActionExecutor instance the agent runs, which means exactly one
# component owns the Paper process, its console channel, and its lifecycle.
# Nothing here builds a shell command or leaves a pipe undrained.
import os
import pathlib
import sys

SERVER_DIR = os.environ.get("SERVER_DIR", "/kaggle/working/minecraft_server")
os.environ["SERVER_DIR"] = SERVER_DIR
os.environ.setdefault("START_TIMEOUT_SECONDS", "600")

for _candidate in (os.environ.get("PANEL_DIR"), "/kaggle/working/rahee-world", "/kaggle/working"):
    if _candidate and os.path.isdir(os.path.join(_candidate, "agent")):
        if _candidate not in sys.path:
            sys.path.insert(0, _candidate)
        break

from agent.actions import action_executor
from agent.lifecycle import lifecycle
from agent.process_info import find_paper_pid

print("=" * 60)
print("STARTING MINECRAFT SERVER")
print("=" * 60)

# Point the executor at the configured server directory so no second process can
# open the same world.
action_executor.server_dir = pathlib.Path(SERVER_DIR)
action_executor.server_jar = action_executor.server_dir / "paper.jar"

_result = action_executor.execute({"action_type": "start_server"})
_state = _result.get("state")
_payload = _result.get("result") or {}

if _state == "success":
    print(f"Server is READY (PID {_payload.get('pid')})")
    print(f"   readiness    : {_payload.get('ready_marker')}")
    print(f"   local address: 127.0.0.1:{os.environ.get('SERVER_PORT', '25565')}")
    print("   Run the Playit.gg cell to get the public address.")
elif _state == "timeout":
    print("Java is running but Paper has not reported readiness yet.")
    print(f"   {_payload.get('warning')}")
    print("   Re-check with the health-check cell; the world may still be generating.")
elif _payload.get("already_running"):
    print(f"A server is already running (PID {_payload.get('pid')}).")
else:
    raise RuntimeError(f"Start failed: {_result.get('error')}")

print("=" * 60)

# Best-effort: if the kernel exits (Ctrl-C, "Stop session"), ask Minecraft to
# save and stop rather than letting the JVM be killed outright.
import atexit
atexit.register(lambda: lifecycle.shutdown())

_LS = lifecycle.status()
print(f"Lifecycle state : {_LS['server_state']} (desired: {_LS['desired_state']})")
print("Crash recovery  : enabled (bounded restarts with backoff)")
print("Keep-alive      : run Cell 15 to keep this kernel alive until you press Stop")
print("=" * 60)

# --- Periodic Google Drive backup (preserved from the original flow) ---------
# Kaggle can end a session without warning, so the world is snapshotted on an
# interval while the server is alive. Manual backup/restore stays in the
# Drive Backup Center cell.
if globals().get("ENABLE_DRIVE_BACKUP"):
    import threading
    import time

    def periodic_backup():
        from drive_backup import GoogleDriveBackup
        while find_paper_pid(SERVER_DIR) is not None:
            try:
                drive = GoogleDriveBackup(dry_run=globals().get("DRY_RUN", False))
                if drive.authenticated:
                    drive.create_backup()
            except Exception as e:
                print(f"Backup error: {e}")
            for _ in range(int(globals().get("BACKUP_INTERVAL_MINUTES", 30)) * 60):
                if find_paper_pid(SERVER_DIR) is None:
                    break
                time.sleep(1)

    threading.Thread(target=periodic_backup, name="rahee-drive-backup",
                     daemon=True).start()
    print("Background Google Drive backup thread started.")
else:
    print("ENABLE_DRIVE_BACKUP is off; Drive backups are manual this session.")

In [ ]:
# Cell 10: Playit.gg Tunnel Setup (Java Edition TCP)
#
# Playit.gg provides free tunneling for game servers without port forwarding.
# The tunnel forwards a public address to 127.0.0.1 on the Minecraft port
# (25565 by default). The local target is configured in the Playit dashboard for
# the tunnel you claim; this cell starts the connector and reports the real
# public address it is given.
#
# This repo does not install or configure Geyser or Floodgate, so Bedrock
# crossplay is out of scope here.
import atexit
import os
import sys
import time

sys.path.insert(0, "/kaggle/working")

print("=" * 60)
print("PLAYIT.GG TUNNEL SETUP")
print("=" * 60)

SERVER_DIR = os.environ.get("SERVER_DIR", "/kaggle/working/minecraft_server")
MINECRAFT_PORT = int(os.environ.get("SERVER_PORT", "25565"))

# The tunnel is only useful once Paper accepts connections. Starting Playit
# first is allowed, but we must say plainly which of the two is not ready.
_server_ready = False
try:
    for _candidate in (os.environ.get("PANEL_DIR"), "/kaggle/working/rahee-world", "/kaggle/working"):
        if _candidate and os.path.isdir(os.path.join(_candidate, "agent")):
            if _candidate not in sys.path:
                sys.path.insert(0, _candidate)
            break
    from agent.process_info import find_paper_pid, find_playit_pid, paper_startup_state
    if find_paper_pid(SERVER_DIR) is None:
        print("WARNING: no Paper server is running yet (start it in Cell 9 first).")
        print("         The tunnel will be up, but players cannot connect yet.")
    elif paper_startup_state(SERVER_DIR)["state"] != "ready":
        print("WARNING: Paper is running but not accepting connections yet.")
    else:
        _server_ready = True
        print(f"Paper is READY on 127.0.0.1:{MINECRAFT_PORT}")
except Exception as _e:
    print(f"Could not verify server readiness ({_e}); continuing with the tunnel.")
    def find_playit_pid():
        return None

from playit_setup import PlayitTunnel

print("\nInitializing Playit tunnel...")
tunnel = PlayitTunnel(dry_run=False)
tunnel.setup_directory()

# Never run two connectors: a duplicate would fight over the same tunnel.
_existing = find_playit_pid()
if _existing:
    print(f"A Playit connector is already running (PID {_existing}); not starting another.")
else:
    token = tunnel._get_token()
    if token:
        print("Token found in Kaggle Secrets (PLAYIT_TOKEN).")
    else:
        print("No PLAYIT_TOKEN found in Kaggle Secrets.")
        print("  1. Sign up at https://playit.gg")
        print("  2. Copy your account token from the dashboard")
        print("  3. Add PLAYIT_TOKEN to Kaggle Secrets, then re-run this cell")
        print("  (The agent prints a claim code you can enter at playit.gg instead.)")

    if not token:
        print("Skipping the agent download until PLAYIT_TOKEN is configured.")
        binary_path = None
    else:
        print("\nDownloading the Playit agent...")
        binary_path = tunnel.download_agent()

        if not binary_path:
            print("WARNING: the Playit agent binary is unavailable. Download it from")
            print("         https://playit.gg/download, upload it, and re-run this cell.")
        else:
            print("\nStarting the Playit agent...")
            if tunnel.start_agent(binary_path):
                atexit.register(lambda: tunnel.stop())

addr = tunnel.get_public_address() if _existing or binary_path else None
if addr:
    hostname, port = addr
    print("\nPLAYIT TUNNEL ACTIVE")
    print(f"   Public hostname: {hostname}")
    print(f"   Public port    : {port}")
    print(f"   Connection     : {hostname}:{port}")
    print("   Java Edition players: Multiplayer -> Add Server -> that address")
elif _existing or binary_path:
    print("\nWARNING: the connector is running but no public address was detected yet.")
    print("         Check https://playit.gg/dashboard for the assigned address.")

print(f"\nTunnel status: {'running' if (_existing or binary_path) else 'not running'} | "
      f"Paper ready: {_server_ready}")
# Register the tunnel with the server lifecycle: the component that owns Paper
# also owns its helpers, so an intentional Stop stops the tunnel too. A crash
# restart leaves it alone (Paper comes back on the same port).
try:
    from agent.lifecycle import lifecycle as _lifecycle
    if _existing or binary_path:
        _lifecycle.register_helper(
            "playit",
            stop=lambda: tunnel.stop(),
            is_running=lambda: find_playit_pid() is not None,
        )
        print("Registered the Playit tunnel with the server lifecycle "
              "(Stop now stops it too).")
except Exception as _reg_err:
    print(f"Could not register the tunnel with the lifecycle ({_reg_err}); "
          f"stop it with tunnel.stop() if needed.")


In [ ]:
# Cell 11: Connection information and health check
#
# Reports what is actually true right now, separately for the server and the
# tunnel. Status comes from the same detector the agent and dashboard use, so
# this cell cannot contradict the panel.
import os
import sys

SERVER_DIR = os.environ.get("SERVER_DIR", "/kaggle/working/minecraft_server")

for _candidate in (os.environ.get("PANEL_DIR"), "/kaggle/working/rahee-world", "/kaggle/working"):
    if _candidate and os.path.isdir(os.path.join(_candidate, "agent")):
        if _candidate not in sys.path:
            sys.path.insert(0, _candidate)
        break

from agent.process_info import (find_paper_pid, find_playit_pid,
                                paper_startup_state)

print("=" * 60)
print("SERVER STATUS & CONNECTION INFO")
print("=" * 60)

# --- configuration actually in effect ---------------------------------------
props_path = os.path.join(SERVER_DIR, "server.properties")
port = int(os.environ.get("SERVER_PORT", "25565"))
props = {}
if os.path.exists(props_path):
    with open(props_path) as f:
        for line in f:
            line = line.strip()
            if "=" in line and not line.startswith("#"):
                k, v = line.split("=", 1)
                props[k.strip()] = v.strip()
    try:
        port = int(props.get("server-port", port))
    except ValueError:
        pass

print(f"\nServer directory: {SERVER_DIR}")
print(f"Level name      : {props.get('level-name', 'world')}")
print(f"Local address   : 127.0.0.1:{port}")
for key in ("max-players", "view-distance", "difficulty", "online-mode", "motd"):
    if key in props:
        print(f"{key:16s}: {props[key]}")

# --- truthful readiness (not just 'a JVM exists') ---------------------------
pid = find_paper_pid(SERVER_DIR)
startup = paper_startup_state(SERVER_DIR, os.path.join(SERVER_DIR, "logs", "latest.log"))

print("\nPaper status:")
if pid is None:
    print("   STOPPED - no Paper process for this directory")
    print("   Start it with Cell 9.")
else:
    print(f"   process   : running (PID {pid})")
    print(f"   readiness : {startup['state']}")
    if startup["state"] == "ready":
        print(f"   last line : {startup['marker']}")
        print(f"   players   : connect to 127.0.0.1:{port} locally")
    elif startup["state"] == "loading":
        print("   Minecraft is still loading; players cannot join yet.")
    elif startup["state"] == "failed":
        print(f"   Paper reported a startup failure: {startup['marker']}")

# --- tunnel status is separate from server status ---------------------------
playit_pid = None
try:
    playit_pid = find_playit_pid()
except Exception:
    pass

public_address = None
try:
    from playit_setup import PlayitTunnel
    public_address = PlayitTunnel(dry_run=False).get_public_address()
except Exception:
    pass

print("\nPlayit.gg tunnel:")
print(f"   connector : {'running (PID %s)' % playit_pid if playit_pid else 'not running'}")
if public_address:
    print(f"   address   : {public_address[0]}:{public_address[1]}")
    print("   Java Edition players: Multiplayer -> Add Server -> that address")
else:
    print("   address   : not detected - see https://playit.gg/dashboard")

if pid is not None and startup["state"] == "ready" and public_address:
    print("\nPlayers can connect now.")
elif pid is not None and startup["state"] == "ready":
    print("\nServer is ready but the tunnel has no address yet; run Cell 10.")
else:
    print("\nNot ready for players yet - see the statuses above.")

# --- recent log ------------------------------------------------------------
log_file = os.path.join(SERVER_DIR, "logs", "latest.log")
print("\nServer log (last 20 lines):")
if os.path.exists(log_file):
    with open(log_file, errors="replace") as f:
        for line in f.readlines()[-20:]:
            print("   " + line.rstrip())
else:
    print("   (no log yet)")

print("\n" + "=" * 60)

In [ ]:
# Cell 12: Google Drive Backup Center — Manual Operations
import os
import sys

print("=" * 60)
print("GOOGLE DRIVE BACKUP CENTER — MANUAL OPERATIONS")
print("=" * 60)

if ENABLE_DRIVE_BACKUP:
    try:
        from drive_backup import GoogleDriveBackup, test_backup_system

        drive = GoogleDriveBackup(dry_run=DRY_RUN)

        if not drive.authenticated:
            drive.authenticate()

        # Show backup status first
        print("\nBACKUP STATUS:")
        backups = drive.list_backups()

        print("\nMANUAL BACKUP OPERATIONS:\n")

        # 1. Create a manual backup now
        print("   1) Create a manual backup now:")
        print("      ```python")
        print("      from drive_backup import GoogleDriveBackup")
        print("      drive = GoogleDriveBackup()  # or GoogleDriveBackup(dry_run=True)")
        print("      drive.create_backup()  # returns backup filename on success")
        print("      ```")

        # 2. Download / retrieve a specific backup metadata + file
        print("\n   2) Retrieve / inspect a specific backup:")
        print("      ```python")
        print("      # List backups first to find an id / name")
        print("      for b in drive.list_backups():")
        print("          print(b['name'], b.get('id'), b.get('size') / (1024*1024), 'MB')")
        print("      # Then inspect by id:")
        print("      drive._get_backup_by_id('BACKUP_ID')")
        print("      ```")

        # 3. Restore a selected backup
        print("\n   3) Restore a selected backup (explicit confirm required):")
        print("      ```python")
        print("      # The restore method requires a manual 'y' confirmation")
        print("      # in an interactive Kaggle session, or force_replace=True")
        print("      drive.restore_backup(backup_id='BACKUP_ID', force_replace=False)")
        print("      ```")
        print("      WARNING: Before restoring, the existing world is preserved as")
        print("               a pre-restore safety backup unless force_replace=True.")

        # 4. Delete a backup
        print("\n   4) Delete a backup (explicit confirm required):")
        print("      ```python")
        print("      # Drive deletion happens inside create_backup()'s cleanup")
        print("      # and via the Google Drive API by file id.")
        print("      # To delete a specific backup by id:")
        print("      from googleapiclient.discovery import build")
        print("      # Re-use the authenticated drive.service if available:")
        print("      if drive.service:")
        print("          drive.service.files().delete(fileId='BACKUP_ID').execute()")
        print("      ```")
        print("      WARNING: Deletion is permanent — confirm the backup id before deleting.")

        print("\nNOTES:\n")
        print("   - Backups are stored in the Drive folder:\n")
        print("     Minecraft-Server-Backups\n")
        print("   - Timestamps, sizes, and status are visible via:\n")
        print("     drive.list_backups()\n")
        print("   - Each backup is a timestamped ZIP of world/,\n")
        print("     world_nether/, world_the_end/ (if present).\n")
        print("   - Credentials are kept in Kaggle Secrets and\n")
        print("     /kaggle/working/.drive_token.json (chmod 600).\n")
        print("   - The existing drive_backup.py module handles\n")
        print("     authentication, uploads, downloads, integrity\n")
        print("     checks, and cleanup.\n")

    except Exception as e:
        print(f"ERROR: {e}")
else:
    print("\nGoogle Drive backup disabled")

print("\n" + "=" * 60)


In [ ]:
# Cell 13: Logs, backup, and shutdown handling
import os
import shutil
import zipfile
from datetime import datetime

SERVER_DIR = os.environ.get("SERVER_DIR", "/kaggle/working/minecraft_server")
BACKUP_DIR = "/kaggle/working/backups"
os.makedirs(BACKUP_DIR, exist_ok=True)

print("=" * 60)
print("LOGS, BACKUP & SHUTDOWN HANDLING")
print("=" * 60)

# ===== GOOGLE DRIVE BACKUP (if enabled) =====
if globals().get("ENABLE_DRIVE_BACKUP"):
    print("\nGOOGLE DRIVE BACKUP:")
    print("   Your world is backed up to Google Drive while the server runs.")
    print("   For a final backup before closing:")
    print("")
    print("     from drive_backup import GoogleDriveBackup")
    print("     drive = GoogleDriveBackup()")
    print("     drive.create_backup()")

# ===== LOCAL BACKUP (fallback) =====
print("\nLOCAL BACKUP (fallback):")
world_dir = os.path.join(SERVER_DIR, "world")
if os.path.isdir(world_dir):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_name = f"world_backup_{timestamp}.zip"
    print(f"   Create one with: !zip -r /kaggle/working/backups/{backup_name} "
          f"{SERVER_DIR}/world/")
    print("   Then download it from the Kaggle Files panel.")
else:
    print("   No world found yet.")

# ===== RESTORATION INSTRUCTIONS =====
print("\nRESTORING FROM GOOGLE DRIVE:")
print("   1. Authenticate with Drive (Cell 3)")
print("   2. List backups: drive.list_backups()")
print("   3. Restore: drive.restore_backup()")
print("   4. Start the server (Cell 9)")

# ===== COMMON ISSUES =====
print("""
TROUBLESHOOTING:

- Drive backup not working:
  - Check credentials in Kaggle Secrets (any of these naming conventions):
    - GDRIVE_CLIENT_ID / GDRIVE_CLIENT_SECRET (n8n style)
    - GOOGLE_OAUTH_CLIENT_ID / GOOGLE_OAUTH_CLIENT_SECRET
    - GOOGLE_CLIENT_ID / GOOGLE_CLIENT_SECRET (generic)
  - Verify the OAuth consent screen is configured in Google Cloud Console
  - Re-run the authentication cell (Cell 3)

- Paper download failed:
  - Re-run Cell 6; it refuses to install a file that fails verification
  - Set PAPER_FORCE_DOWNLOAD=1 only when you intend to replace paper.jar

- Server won't start:
  - Check Java is installed (Cell 1 must report Java 21)
  - Verify the EULA was accepted (eula.txt must contain 'eula=true')
  - Check available RAM - lower JAVA_XMX if you see OOM errors
  - Read the tail of logs/latest.log for the real error

- Can't connect from outside:
  - Make sure the Playit connector is running (Cell 10)
  - Check the Playit dashboard for the assigned address
  - Verify PLAYIT_TOKEN is set correctly

- Lag / performance:
  - Lower VIEW_DISTANCE to 4-6
  - Lower MAX_PLAYERS
  - Paper is already configured for performance

- World not saving:
  - Stop the server with the dashboard stop button (Paper's own 'stop')
  - Run a final Drive backup before closing the notebook
""")

# ===== GRACEFUL SHUTDOWN =====
print("SHUTDOWN PROCEDURE:")
print("   1. Press Stop in the dashboard (Cell 15 returns when it completes)")
print("   2. Confirm logs/latest.log ends with a clean shutdown")
print("   3. Run a final Drive backup (Cell 13)")
print("   4. Stop the notebook kernel")
print()
print("CRASH RECOVERY:")
print("   - The supervisor (Cell 15) detects an unexpected Minecraft exit")
print("   - Exit code and log tail are recorded; restarts use exponential backoff")
print("   - After the bounded attempts are used up the dashboard shows CRASHED")
print("   - An intentional Stop is never followed by a restart")

print("\nIMPORTANT WARNINGS:")
print("   - Kaggle may terminate sessions without warning")
print("   - Periodic backups help but are not guaranteed")
print("   - Always do a manual backup before an important session")
print("   - World data is lost if the session ends without a backup")

print("\n" + "=" * 60)
print("SETUP COMPLETE")
print("=" * 60)
print("NEXT STEPS:")
print("   1. Cell 9 starts the server and waits for real readiness")
print("   2. Cell 10 starts the Playit tunnel and prints the public address")
print("   3. Share that address with your Java Edition players")
print("   4. Cell 14 runs the dashboard panel in this kernel")
print("=" * 60)

In [ ]:
# Cell 14: Start the RaheeCraft web panel in this kernel
#
# The panel is the dashboard/API. In production it is deployed separately (see
# backend/render.yaml); this cell runs it inside the Kaggle session so you can
# reach it through whatever tunnel you use for the *web* port.
#
# That is a different tunnel from the Playit.gg Minecraft tunnel in Cell 10.
# Do NOT expose the agent endpoint (/api/agent/*) publicly; only the web app
# goes through any web tunnel.
#
# Secrets read from Kaggle Secrets:
#   DEFAULT_ADMIN_USER / DEFAULT_ADMIN_PASS - initial admin login
#   CORS_ORIGIN                            - the origin you serve the panel on
#   AGENT_ID / AGENT_SECRET / BACKEND_URL  - only if the panel also runs an agent
#   CLOUDFLARE_TUNNEL_TOKEN                - token of the Cloudflare tunnel that
#                                            publishes this panel (see below)
#   CLOUDFLARE_TUNNEL_ID / CLOUDFLARE_PANEL_HOSTNAME - informational only: the
#                                            token authenticates the connector,
#                                            the public hostname lives in the
#                                            Cloudflare dashboard
import atexit
import os
import signal
import subprocess
import sys
import time

PANEL_DIR = os.environ.get("PANEL_DIR", "/kaggle/working/rahee-world")
os.environ.setdefault("PANEL_DIR", PANEL_DIR)
os.environ.setdefault("SERVER_DIR", "/kaggle/working/minecraft_server")
os.environ.setdefault("PORT", "8080")

for _key in ("DEFAULT_ADMIN_USER", "DEFAULT_ADMIN_PASS", "CORS_ORIGIN",
             "AGENT_ID", "AGENT_SECRET", "BACKEND_URL",
             "CLOUDFLARE_TUNNEL_TOKEN", "CLOUDFLARE_TUNNEL_ID",
             "CLOUDFLARE_PANEL_HOSTNAME"):
    if not os.environ.get(_key):
        try:
            from kaggle_secrets import UserSecretsClient
            _value = UserSecretsClient().get_secret(_key)
            if _value:
                os.environ[_key] = _value
        except Exception:
            pass

port = int(os.environ.get("PORT", "8080"))
print(f"Starting the RaheeCraft web panel on 127.0.0.1:{port} ...")

_proc = [None]
_panel_log = [None]


def _panel_log_handle():
    """Open the panel's log file.

    The panel is a long-lived child: an unread stdout PIPE fills its 64 KiB
    buffer and freezes the panel (the same failure that used to hang the Paper
    JVM). Its output goes to a file, read back only when something goes wrong.
    """
    log_dir = os.path.join(PANEL_DIR, "logs")
    os.makedirs(log_dir, exist_ok=True)
    _panel_log[0] = open(os.path.join(log_dir, "panel.log"), "a",
                         encoding="utf-8", buffering=1)
    return _panel_log[0]


def _panel_log_tail(lines=40):
    try:
        path = os.path.join(PANEL_DIR, "logs", "panel.log")
        with open(path, encoding="utf-8", errors="replace") as fh:
            return [l.rstrip() for l in fh.readlines()[-lines:]]
    except Exception as exc:
        return [f"(could not read the panel log: {exc})"]


def _start_panel():
    script = os.path.join(PANEL_DIR, "run_backend.py")
    if not os.path.exists(script):
        script = os.path.join(PANEL_DIR, "backend", "app.py")
    if not os.path.exists(script):
        raise RuntimeError(f"panel entry point not found under {PANEL_DIR}")
    return subprocess.Popen(
        [sys.executable, script],
        cwd=PANEL_DIR, env=dict(os.environ),
        stdout=_panel_log_handle(), stderr=subprocess.STDOUT,
    )


def _wait_for_panel(timeout=90):
    import urllib.request
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            # /health is public; /api/status needs a session and returns 401,
            # which made this readiness check report a false failure.
            with urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=2) as r:
                if r.status == 200:
                    return True
        except Exception:
            pass
        time.sleep(2)
    return False


try:
    _proc[0] = _start_panel()
    print(f"Panel process started (PID {_proc[0].pid})")
    if _wait_for_panel():
        print(f"Web panel is reachable at http://127.0.0.1:{port}")
    else:
        print("WARNING: the web panel did not become reachable in time.")
        for _line in _panel_log_tail():
            print("  |", _line)
except Exception as e:
    print(f"ERROR starting the panel: {e}")


# --- Public access: the existing Cloudflare tunnel ---------------------------
# The panel listens on loopback *inside this kernel*, so the connector has to
# run here as well: nothing outside the kernel can reach 127.0.0.1:8080.
# A token connector takes its ingress rule (public hostname -> local service)
# from the Cloudflare dashboard, so this cell configures no DNS and no
# hostname. The token reaches cloudflared through the environment only, never
# on a command line, and it is redacted from all captured output.
_tunnel = [None]
TUNNEL_ID = os.environ.get("CLOUDFLARE_TUNNEL_ID", "")
PANEL_HOSTNAME = os.environ.get("CLOUDFLARE_PANEL_HOSTNAME", "")

if _proc[0] is not None and _proc[0].poll() is None:
    print()
    try:
        from cloudflare_tunnel import CloudflareTunnel
        _tunnel[0] = CloudflareTunnel(tunnel_id=TUNNEL_ID or None,
                                      hostname=PANEL_HOSTNAME or None,
                                      local_port=port)
        if not _tunnel[0].has_token():
            print("Cloudflare tunnel: skipped - no CLOUDFLARE_TUNNEL_TOKEN here.")
            print("  1. Cloudflare: Zero Trust -> Networks -> Tunnels -> the panel")
            print("     tunnel -> Install connector -> copy the token.")
            print("  2. Kaggle: Add-ons -> Secrets -> label CLOUDFLARE_TUNNEL_TOKEN,")
            print("     attach it to THIS notebook, then re-run this cell.")
            print(f"  Tunnel id : {_tunnel[0].tunnel_id}")
            print(f"  Hostname  : {_tunnel[0].hostname}")
            print("  A token connector cannot be authenticated with the id alone.")
        else:
            print("Starting the Cloudflare tunnel connector...")
            if not _tunnel[0].start():
                print(f"  connector did not start: {_tunnel[0].last_error}")
            else:
                _result = _tunnel[0].wait_for_connection(timeout=90)
                _state = _tunnel[0].status()
                _label = "CONNECTED" if _result["connected"] else "NOT connected"
                print(f"  tunnel       : {_label}")
                print(f"  reason       : {_result['reason']}")
                print(f"  tunnel id    : {_state['tunnel_id']}")
                print(f"  public host  : https://{_state['hostname']}")
                print("                 (the Zero Trust dashboard must route it to")
                print("                  http://localhost:8080 on this connector)")
                print(f"  local panel  : {_state['local_service']} "
                      f"answering={_state['local_service_ok']}")
                print(f"  connector log: {_state['log_file']}")
                for _line in _result.get("tail", []):
                    print("  |", _line)
                if _result["connected"]:
                    # A connected connector is not a working URL: check the
                    # whole path (edge -> tunnel -> connector -> panel).
                    _public = _tunnel[0].verify_public_service()
                    _verdict = "OK" if _public["ok"] else "not serving yet"
                    print(f"  public check : {_public['url']} -> {_verdict} "
                          f"({_public['detail']})")
                    if not _public["ok"]:
                        print("                 the connector is up, but that hostname")
                        print("                 must exist in the Zero Trust dashboard and")
                        print("                 route to http://localhost:8080 here.")
                if _result["connected"] and not _state["local_service_ok"]:
                    print("  WARNING: the tunnel is up but the panel is not "
                          "answering locally.")
                if not _result["connected"]:
                    print("  The panel is reachable inside this kernel only.")
    except Exception as _tunnel_err:
        print(f"Cloudflare tunnel: not started ({_tunnel_err})")

def _cleanup():
    # The panel tunnel is deliberately NOT registered with the Minecraft
    # lifecycle: it publishes the *panel*, and killing it whenever Minecraft
    # stops would remove the very controls needed to start Minecraft again.
    if _tunnel[0] is not None:
        try:
            _tunnel[0].stop()
        except Exception:
            pass
    p = _proc[0]
    if p and p.poll() is None:
        try:
            p.send_signal(signal.SIGTERM)
            try:
                p.wait(timeout=5)
            except Exception:
                p.kill()
        except Exception:
            pass
    if _panel_log[0] is not None:
        try:
            _panel_log[0].close()
        except Exception:
            pass


atexit.register(_cleanup)
print("Panel startup cell finished. Keep this kernel running while the panel is needed.")

In [ ]:
# Cell 15: Runtime supervisor — keeps RaheeCraft running until you press Stop
#
# This cell BLOCKS on purpose. While it is in its execution loop:
#   * this kernel stays alive, so the agent thread and Paper keep running;
#   * the lifecycle manager checks the server every few seconds (no busy loop);
#   * an unexpected exit is detected, reported, and recovered within a bounded
#     budget of restarts with exponential backoff;
#   * pressing Stop in the dashboard ends the server AND returns this cell;
#   * the Playit tunnel is stopped along with the server (Cell 10 registered it).
#
# It owns no process of its own: all state comes from the one lifecycle manager,
# so the notebook and the agent can never fight over the server.
#
# Kaggle ends sessions on its own schedule (idle timeout, quota, or a manual
# "Stop session"). Nothing here can prevent that, and this cell does not pretend
# otherwise: when the runtime goes away, the server goes with it. That is what
# the periodic Drive backup is for.
#
# Optional env knobs:
#   HEALTH_PRINT_SECONDS   how often a health line is printed (default 60)
#   SUPERVISOR_MAX_SECONDS stop supervising after N seconds (default: no limit)
import os
import sys

SERVER_DIR = os.environ.get("SERVER_DIR", "/kaggle/working/minecraft_server")
os.environ["SERVER_DIR"] = SERVER_DIR

for _candidate in (os.environ.get("PANEL_DIR"),
                   "/kaggle/working/rahee-world", "/kaggle/working"):
    if _candidate and os.path.isdir(os.path.join(_candidate, "agent")):
        if _candidate not in sys.path:
            sys.path.insert(0, _candidate)
        break

from agent.lifecycle import lifecycle
from agent.status import get_agent_status

HEALTH_PRINT_SECONDS = int(os.environ.get("HEALTH_PRINT_SECONDS", "60"))
_max_raw = os.environ.get("SUPERVISOR_MAX_SECONDS", "")
MAX_SECONDS = int(_max_raw) if _max_raw.isdigit() and int(_max_raw) > 0 else None

print("=" * 60)
print("RUNTIME SUPERVISOR")
print("=" * 60)
print(f"Server directory : {SERVER_DIR}")
print(f"Health interval  : {lifecycle.health_interval():.0f}s")
print(f"Print every      : {HEALTH_PRINT_SECONDS}s")
print(f"Time budget      : {'none (runs until Stop)' if MAX_SECONDS is None else str(MAX_SECONDS) + 's'}")
print("Press Stop in the dashboard to end the server and this cell.")
print("=" * 60)

_last_print = [0.0]


def _report(status):
    """Print a real health line. Never assumes the server is fine."""
    import time as _time
    if _time.time() - _last_print[0] < HEALTH_PRINT_SECONDS:
        return
    _last_print[0] = _time.time()

    helpers = status.get("helpers") or {}
    playit = "n/a"
    if "playit" in helpers:
        playit = "running" if helpers["playit"].get("running") else "not running"

    players = "n/a"
    try:
        snapshot = get_agent_status().get("player_snapshot") or {}
        online = snapshot.get("online")
        if isinstance(online, list):
            players = len(online)
    except Exception:
        pass

    print(f"[supervisor] state={status['server_state']} "
          f"desired={status['desired_state']} pid={status['paper_pid']} "
          f"uptime={status['uptime_seconds']}s "
          f"restarts={status['restart_attempts']} playit={playit} "
          f"players={players}")
    if status.get("restart_scheduled_in") is not None:
        print(f"[supervisor] restart scheduled in "
              f"{status['restart_scheduled_in']}s")
    if status.get("last_error"):
        print(f"[supervisor] last error: {status['last_error']}")


try:
    outcome = lifecycle.wait_while_running(on_tick=_report, max_seconds=MAX_SECONDS)
except KeyboardInterrupt:
    # Interrupting the cell stops watching, NOT the server. Say so plainly.
    outcome = {"reason": "cell interrupted (server left running)",
               "seconds": None}

final = lifecycle.status()
print()
print("=" * 60)
print(f"SUPERVISOR EXITED: {outcome.get('reason')} "
      f"({outcome.get('seconds')}s)")
print(f"  server state : {final['server_state']}")
print(f"  running      : {final['paper_running']} (pid {final['paper_pid']})")
if final.get("last_error"):
    print(f"  last error   : {final['last_error']}")
print("=" * 60)

if final["server_state"] == "CRASHED":
    print("The server is NOT running (CRASHED).")
    print("  - Read the error above and the console tail in the dashboard")
    print("  - Fix the cause (jar, Java version, memory), then press Start")
elif final["paper_running"] and final["desired_state"] == "RUNNING":
    print("The server is STILL RUNNING; this cell only stopped watching.")
    print("Re-run this cell to resume the health output.")
else:
    print("The server is stopped and the world was saved by Minecraft's own")
    print("`stop`. Run a final Drive backup (Cell 13) before ending the session.")
